In [1]:
import os

os.environ['HTTP_PROXY'] = "http://127.0.0.1:1081"
os.environ['HTTPS_PROXY'] = "http://127.0.0.1:1081"

In [4]:
from datasets import load_dataset
import random
from transformers import AutoTokenizer

In [5]:

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
ds = load_dataset("open-r1/OpenR1-Math-220k", "default")

README.md:   0%|          | 0.00/5.13k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

train-00000-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

train-00001-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

train-00002-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

train-00006-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

train-00007-of-00010.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

train-00008-of-00010.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

train-00009-of-00010.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93733 [00:00<?, ? examples/s]

In [7]:
item = random.choice(ds["train"])
print(item)

{'problem': '192*. The sequence $\\left(x_{n}\\right)$ is defined by the conditions:\n\n$$\nx_{1}=\\frac{1}{2}, \\quad x_{n+1}=\\sqrt{\\frac{1-\\sqrt{1-x_{n}^{2}}}{2}}\n$$\n\nFind a formula for $x_{n}$.', 'solution': '$\\triangle$ From the condition, it is clear that $0 \\leq x_{n} \\leq 1$. Let $x_{n}=\\sin \\alpha$, where $\\alpha \\in\\left[0 ; \\frac{\\pi}{2}\\right]$. We get:\n\n$$\nx_{n+1}=\\sqrt{\\frac{1-\\sqrt{1-\\sin ^{2} \\alpha}}{2}}=\\sqrt{\\frac{1-\\cos \\alpha}{2}}=\\sin \\frac{\\alpha}{2}\n$$\n\nThe argument of $x_{n+1}$ under the sine sign turned out to be twice as small as that of $x_{n}$. Then, since\n\n$$\nx_{1}=\\sin \\frac{\\pi}{6}=\\sin \\frac{\\pi}{3 \\cdot 2}\n$$\n\nthe argument of $x_{2}$ under the sine sign is $\\frac{\\pi}{3 \\cdot 2^{2}}, \\mathrm{y} x_{3}-\\frac{\\pi}{3 \\cdot 2^{3}}$, generally, the argument of $x_{n}-\\frac{\\pi}{3 \\cdot 2^{n}}$.\n\nAnswer: $x_{n}=\\sin \\frac{\\pi}{3 \\cdot 2^{n}}$.', 'answer': 'x_{n}=\\sin\\frac{\\pi}{3\\cdot2^{n}}', '

In [8]:
print(item.keys())

dict_keys(['problem', 'solution', 'answer', 'problem_type', 'question_type', 'source', 'uuid', 'is_reasoning_complete', 'generations', 'correctness_math_verify', 'correctness_llama', 'finish_reasons', 'correctness_count', 'messages'])


In [11]:
print(len(item['generations']))
print(item['generations'][0])
print('\n\n--\n\n')
print(item['generations'][1])

2
<think>
Okay, let's try to figure out this sequence problem. So, the sequence is defined by x₁ = 1/2, and then each subsequent term is xₙ₊₁ = sqrt[(1 - sqrt(1 - xₙ²))/2]. The question is to find a formula for xₙ. Hmm, okay, let's start by writing down the first few terms to see if I can spot a pattern.

Starting with x₁ = 1/2. Then x₂ is sqrt[(1 - sqrt(1 - (1/2)²))/2]. Let me compute that step by step. First, compute x₁²: (1/2)² = 1/4. Then, sqrt(1 - 1/4) = sqrt(3/4) = sqrt(3)/2. So, 1 - sqrt(1 - x₁²) is 1 - sqrt(3)/2. Then divide that by 2: (1 - sqrt(3)/2)/2 = (2 - sqrt(3))/4. Taking the square root gives x₂ = sqrt[(2 - sqrt(3))/4] = sqrt(2 - sqrt(3))/2. Hmm, okay. Let me note that sqrt(2 - sqrt(3)) might be a familiar expression. Wait, maybe related to trigonometric functions? Let me recall that sin(15°) = sin(π/12) = (sqrt(6) - sqrt(2))/4, and cos(15°) = (sqrt(6) + sqrt(2))/4. Alternatively, sqrt(2 - sqrt(3))/2 is actually equal to sin(π/12) or something? Let me check: If I square

In [ ]:
def extract_reason_traces(item, k=1):
    if 'generations' not in item or len(item['generations']) < k:
        return []
    
    samples = random.choices(item['generations'], k=k)
    return samples



In [ ]:



def create_incoherent_sample(
    text, tokenizer, location="random", intensity=0.5, operations=["swap", "substitute", "delete", "repeat", "insert_noise"]
):
    """
    Create an incoherent sample from coherent text with tokenizer-based transformations.
    """
    # Tokenize the text
    tokens = tokenizer.encode(text, add_special_tokens=True)
    total_tokens = len(tokens)

    if total_tokens <= 1:
        return text

    # Determine segment to transform
    segment_size = int(total_tokens * intensity)
    segment_size = max(1, segment_size)

    if location == "beginning":
        start_idx = 0
        end_idx = min(segment_size, total_tokens)
    elif location == "end":
        start_idx = max(0, total_tokens - segment_size)
        end_idx = total_tokens
    elif location == "middle":
        mid_point = total_tokens // 2
        half_segment = segment_size // 2
        start_idx = max(0, mid_point - half_segment)
        end_idx = min(total_tokens, mid_point + half_segment)
    elif location == "random":
        if total_tokens <= segment_size:
            start_idx = 0
            end_idx = total_tokens
        else:
            start_idx = random.randint(0, total_tokens - segment_size)
            end_idx = start_idx + segment_size
    else:  # "full"
        start_idx = 0
        end_idx = total_tokens

    # Work on a copy of tokens
    modified_tokens = tokens.copy()
    vocab_size = tokenizer.vocab_size

    # Define indices to modify with higher intensity within segment
    indices_to_modify = list(range(start_idx, end_idx))
    num_to_modify = max(1, int(total_tokens * intensity))  # Apply intensity to full length

    if num_to_modify > len(indices_to_modify):
        num_to_modify = len(indices_to_modify)
    selected_indices = random.sample(indices_to_modify, num_to_modify)

    # Noise characters to simulate "garbagy" text
    noise_chars = "!@#$%^&*()_+-=[]{}|;:,.<>?~`"
    noise_tokens = [
        tokenizer.encode(c, add_special_tokens=False)[0] for c in noise_chars if tokenizer.encode(c, add_special_tokens=False)
    ]

    for idx in selected_indices[:]:  # Copy to avoid modification during iteration
        if idx >= len(modified_tokens):
            continue
        operation = random.choice(operations)

        if operation == "swap" and idx < len(modified_tokens) - 1:
            # Swap with a random token (not just adjacent)
            swap_idx = random.randint(0, len(modified_tokens) - 1)
            while swap_idx == idx:  # Ensure it's not the same token
                swap_idx = random.randint(0, len(modified_tokens) - 1)
            modified_tokens[idx], modified_tokens[swap_idx] = modified_tokens[swap_idx], modified_tokens[idx]

        elif operation == "substitute":
            # Replace with a random token or a rare one to increase nonsense
            modified_tokens[idx] = random.randint(0, vocab_size - 1)

        elif operation == "delete" and len(modified_tokens) > 1:
            # Delete with a cap to preserve some structure
            if random.random() < 0.5:  # 50% chance to delete
                modified_tokens.pop(idx)
                selected_indices = [i if i < idx else i - 1 for i in selected_indices if i != idx]

        elif operation == "repeat":
            # Repeat token multiple times for gibberish effect
            repeat_count = random.randint(1, 3)
            for _ in range(repeat_count):
                modified_tokens.insert(idx + 1, modified_tokens[idx])
            selected_indices = [i if i <= idx else i + repeat_count for i in selected_indices]

        elif operation == "insert_noise":
            # Insert random symbols or noise tokens
            noise_token = random.choice(noise_tokens)
            modified_tokens.insert(idx, noise_token)
            selected_indices = [i if i < idx else i + 1 for i in selected_indices]

    # Decode with minimal cleanup to preserve noise
    transformed_text = tokenizer.decode(modified_tokens, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    return transformed_text


def create_training_dataset(positive_samples, tokenizer, locations=None, intensities=None):
    """
    Create training dataset with balanced positive and negative samples.
    """
    if locations is None:
        locations = ["beginning", "middle", "end", "random", "full"]
    if intensities is None:
        intensities = [0.3, 0.5, 0.7, 0.9]  # Higher intensities for more noise

    dataset = []

    # Positive samples
    for sample in positive_samples:
        dataset.append((sample, 1))  # 1 = coherent

    # Negative samples
    for sample in positive_samples:
        location = random.choice(locations)
        intensity = random.choice(intensities)
        incoherent_sample = create_incoherent_sample(sample, tokenizer, location, intensity)
        dataset.append((incoherent_sample, 0))  # 0 = incoherent

    random.shuffle(dataset)
    return dataset


# # Example usage
# if __name__ == "__main__":
#     tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
#     positive_samples = ["This is a coherent sentence.", "I enjoy training models."]
#     dataset = create_training_dataset(positive_samples, tokenizer)
#     for text, label in dataset:
#         print(f"Text: {text} | Label: {label}")

In [15]:

import os

os.environ['HTTP_PROXY'] = "http://127.0.0.1:1081"
os.environ['HTTPS_PROXY'] = "http://127.0.0.1:1081"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Assume we already have positive samples
positive_samples = [
    # Your collection of coherent texts
    """To solve this problem, let's first calculate the total amount of food the adult cats will consume. Sidney has 3 adult cats, and each cat eats 1 can per day. Over a period of 7 days, the total amount of food required for the adult cats is:

<think> 
Adult cats: 3 cats * 1 can per day * 7 days = 21 cans
</think>

Next, let's compute the total amount of food the kittens will consume. Sidney has 4 kittens, and each kitten eats 3/4 of a can per day. For 7 days, the total amount of food required for the kittens is:

<think> 
Kittens: 4 kittens * 3/4 can per day * 7 days = 4 * (3/4) * 7 = 3 * 7 = 21 cans
</think>

Now, let's determine the total amount of food required for all the cats (adult and kitten) over 7 days. Adding the food requirements for the adult cats and the kittens, we get:

<think> 
Total food required: 21 cans (adult cats) + 21 cans (kittens) = 42 cans
</think>

Sidney currently has 7 cans of cat food. To find the number of additional cans needed, we compare the 7 cans she owns with the 42 cans she needs:

<answer>
""",
    """To determine Maximoff's total monthly bill after the increase, we need to follow these steps:

1. **Calculate the increase in the monthly bill**: 
   The increase is 30% of the original monthly bill. The original monthly bill is $60.

   <think>To find 30% of $60, we can use the formula: increase = (percentage increase / 100) * original amount. In this case, the percentage increase is 30% and the original amount is $60.</think>

   <answer>
   <math>
   30\% \text{ of } \$60 = \left(\frac{30}{100}\right) \times 60 = 18
   </math>
   </answer>

   Therefore, the increase in the monthly bill is $18.

2. **Calculate the new total monthly bill**:
   The new total monthly bill is the original bill plus the increase.

   <think>To find the new total bill, we add the increase to the original bill: total new bill = original bill + increase.</think>

   <answer>
   $60 + $18 = \$78$
   </answer>     
   
Thus, Maximoff's total monthly bill after starting to use at home, after a $18 increase, works outto $\boxed{78}$.""",
]



dataset = create_training_dataset(positive_samples, tokenizer)
for text, label in dataset:
    print(f"Text: {text} | Label: {label}")
    print('\n\n--\n\n')


Text: To solve this problem, let's first calculate the total amount of food the adult cats will consume. Sidney has 3 adult cats, and each cat eats 1 can per day. Over a period of 7 days, the total amount of food required for the adult cats is:

<think> 
Adult cats: 3 cats * 1 can per day * 7 days = 21 cans
</think>

Next, let's compute the total amount of food the kittens will consume. Sidney has 4 kittens, and each kitten eats 3/4 of a can per day. For 7 days, the total amount of food required for the kittens is:

<think> 
Kittens: 4 kittens * 3/4 can per day * 7 days = 4 * (3/4) * 7 = 3 * 7 = 21 cans
</think>

Now, let's determine the total amount of food required for all the cats (adult and kitten) over 7 days. Adding the food requirements for the adult cats and the kittens, we get:

<think> 
Total food required: 21 cans (adult cats) + 21 cans (kittens) = 42 cans
</think>

Sidney currently has 7 cans of cat food. To find the number of additional cans needed, we compare the 7 cans s